# Vibe Quickstart Notebook

프로젝트 루트에서 바로 실행 가능한 예시 노트북입니다.
- 백테스트 결과 CSV 로드
- 기본 통계 확인
- 자산곡선 시각화

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.')
curve_path = ROOT / 'backtest_portfolio_curve.csv'
trade_csv = ROOT / 'dynamic_trade_log.csv'

print('workspace:', ROOT.resolve())
print('curve csv exists:', curve_path.exists())
print('trade log exists:', trade_csv.exists())

In [ ]:
if curve_path.exists():
    eq = pd.read_csv(curve_path)
    if 'date' in eq.columns:
        eq['date'] = pd.to_datetime(eq['date'])
    elif 'Date' in eq.columns:
        eq['Date'] = pd.to_datetime(eq['Date'])
    print('equity rows:', len(eq))
    display(eq.head())
else:
    print('backtest_portfolio_curve.csv 파일이 없습니다. 백테스트를 먼저 실행하세요.')

In [ ]:
if curve_path.exists():
    plot_df = pd.read_csv(curve_path)
    x_col = 'date' if 'date' in plot_df.columns else ('Date' if 'Date' in plot_df.columns else None)
    y_col = 'Total' if 'Total' in plot_df.columns else (plot_df.columns[1] if len(plot_df.columns) > 1 else None)

    if x_col and y_col:
        plot_df[x_col] = pd.to_datetime(plot_df[x_col])
        plt.figure(figsize=(12, 4))
        plt.plot(plot_df[x_col], plot_df[y_col], linewidth=1.8)
        plt.title('Backtest Portfolio Curve')
        plt.xlabel('Date')
        plt.ylabel('Equity')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print('곡선 CSV 컬럼 형식을 인식하지 못했습니다:', list(plot_df.columns))

In [ ]:
if trade_csv.exists():
    tr = pd.read_csv(trade_csv)
    print('trade rows:', len(tr))
    if {'Action', 'Return'}.issubset(tr.columns):
        exits = tr[tr['Action'].isin(['SELL', 'HOLD'])].copy()
        exits['Return'] = pd.to_numeric(exits['Return'], errors='coerce').fillna(0)
        win_rate = (exits['Return'] > 0).mean() * 100 if len(exits) else 0
        print(f'exit trades: {len(exits)}')
        print(f'win rate: {win_rate:.2f}%')
        print(f"avg return: {exits['Return'].mean():.2f}%")
    display(tr.head())
else:
    print('dynamic_trade_log.csv 파일이 없습니다. 동적 백테스트를 먼저 실행하세요.')